# 🏗️ PHASE 1: SUPREME VENTURES CORPORATE DECK
## Focus: Exact Structure, Fonts, and Positioning (Cover & Table)

In [ ]:
!pip install python-pptx pydantic -q

In [ ]:
import json
import base64
import os
from typing import List, Optional, Literal, Union, Dict, Any
from pydantic import BaseModel
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from IPython.display import display, HTML

# ==========================================
# 🎨 1. EXACT STYLE CONFIGURATION
# ==========================================
# Colors extracted from your screenshots
class Colors:
    BG = [242, 242, 242]         # #F2F2F2 (Light Grey Background)
    TITLE = [0, 0, 0]            # Black
    ACCENT_BANNER = [225, 226, 246] # Lavender (Banner Box)
    TABLE_HEADER = [250, 250, 250]  # Very light grey for headers
    TABLE_BORDER = [220, 220, 220]
    STATUS_GREEN = [0, 153, 51]   # Corporate Green
    STATUS_BLUE = [0, 112, 192]   # Corporate Blue
    STATUS_RED = [192, 0, 0]

class Fonts:
    SERIF = "Times New Roman"    # For Titles & Headers
    SANS = "Arial"               # For Body & Table Content

def get_rgb(c): return RGBColor(c[0], c[1], c[2])

# ==========================================
# 📝 2. JSON DATA MODELS
# ==========================================

class SlideCover(BaseModel):
    type: Literal["cover"]
    title: str
    subtitle: str
    version_date: str
    logo_left: str  # e.g., "image1"
    logo_right: str # e.g., "image2"

class SlideTable(BaseModel):
    type: Literal["table"]
    title: str
    banner_title: str
    banner_text: str
    columns: List[str]
    rows: List[Dict[str, str]]

class PresentationConfig(BaseModel):
    filename: str
    slides: List[Union[SlideCover, SlideTable]]

# ==========================================
# 🖌️ 3. RENDERER ENGINE (NO EMOJIS, EXACT LAYOUT)
# ==========================================

def add_placeholder_image(slide, path_key, left, top, height, width=None):
    """Draws a grey box if image is missing, or the image if present"""
    if os.path.exists(path_key):
        pic = slide.shapes.add_picture(path_key, left, top, height=height)
        if width: pic.width = width # Optional override
        return pic
    else:
        # Placeholder Box
        w = width if width else height * 2
        box = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, w, height)
        box.fill.solid()
        box.fill.fore_color.rgb = RGBColor(200, 200, 200)
        box.line.color.rgb = RGBColor(150, 150, 150)
        p = box.text_frame.paragraphs[0]
        p.text = f"IMG: {path_key}"
        p.font.size = Pt(10)
        p.font.color.rgb = RGBColor(50, 50, 50)
        return box

def render_cover(prs, data: SlideCover):
    slide = prs.slides.add_slide(prs.slide_layouts[6]) # Blank Layout
    
    # Background
    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = get_rgb(Colors.BG)

    # 1. Logo Left (Supreme) - Positioned per Screenshot 1
    # Lower down (~2.5 inches from top)
    add_placeholder_image(slide, data.logo_left, Inches(0.5), Inches(2.2), Inches(1.2))

    # 2. Logo Right (New Fields) - Top Right Corner
    pic_right = add_placeholder_image(slide, data.logo_right, Inches(10), Inches(0.2), Inches(0.6))
    # Align right manually if it's a shape or pic
    pic_right.left = int(Inches(13.33) - pic_right.width - Inches(0.5))

    # 3. Main Title - Large Serif Black
    tb = slide.shapes.add_textbox(Inches(0.5), Inches(4.0), Inches(12), Inches(1.5))
    p = tb.text_frame.paragraphs[0]
    p.text = data.title
    p.font.name = Fonts.SERIF
    p.font.size = Pt(44)
    p.font.bold = True
    p.font.color.rgb = get_rgb(Colors.TITLE)

    # 4. Subtitle + Date
    sb = slide.shapes.add_textbox(Inches(0.5), Inches(5.2), Inches(12), Inches(1.5))
    p = sb.text_frame.paragraphs[0]
    p.text = data.subtitle + "\n" + data.version_date
    p.font.name = Fonts.SERIF
    p.font.size = Pt(16)
    p.font.color.rgb = get_rgb(Colors.TITLE)


def render_table_slide(prs, data: SlideTable):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    
    # Background
    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = get_rgb(Colors.BG)

    # 1. Main Header
    h1 = slide.shapes.add_textbox(Inches(0.5), Inches(0.4), Inches(12), Inches(0.8))
    p = h1.text_frame.paragraphs[0]
    p.text = data.title
    p.font.name = Fonts.SERIF
    p.font.size = Pt(36)
    p.font.bold = True
    p.font.color.rgb = get_rgb(Colors.TITLE)

    # 2. Lavender Banner (The "Proposed Project Timeline" box)
    banner_y = 1.3
    banner_h = 1.0
    
    # Bold Title inside banner area (Text outside box or top line)
    bt = slide.shapes.add_textbox(Inches(0.5), Inches(banner_y), Inches(12), Inches(0.4))
    p = bt.text_frame.paragraphs[0]
    p.text = data.banner_title
    p.font.name = Fonts.SERIF
    p.font.bold = True
    p.font.size = Pt(14)
    
    # The Box itself
    box = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(banner_y + 0.4), Inches(12.33), Inches(0.8))
    box.fill.solid()
    box.fill.fore_color.rgb = get_rgb(Colors.ACCENT_BANNER)
    box.line.color.rgb = RGBColor(200, 200, 200)
    box.line.width = Pt(0.5)
    
    # Text inside box
    p = box.text_frame.paragraphs[0]
    p.text = data.banner_text
    p.font.name = Fonts.SERIF
    p.font.size = Pt(11)
    p.font.color.rgb = get_rgb(Colors.TITLE)
    p.alignment = PP_ALIGN.LEFT
    box.text_frame.margin_left = Inches(0.1)

    # 3. The Table
    table_y = banner_y + 1.5
    rows = len(data.rows) + 1
    cols = len(data.columns)
    shape = slide.shapes.add_table(rows, cols, Inches(0.5), Inches(table_y), Inches(12.33), Inches(0.5*rows))
    table = shape.table

    # Headers
    for i, col_name in enumerate(data.columns):
        cell = table.cell(0, i)
        cell.text = col_name
        cell.fill.solid()
        cell.fill.fore_color.rgb = get_rgb(Colors.TABLE_HEADER)
        
        p = cell.text_frame.paragraphs[0]
        p.font.name = Fonts.SERIF
        p.font.bold = True
        p.font.size = Pt(12)
        p.font.color.rgb = get_rgb(Colors.TITLE)

    # Rows
    for r, row_data in enumerate(data.rows, start=1):
        # Zebra Striping
        row_fill = [255, 255, 255] if r % 2 == 0 else [250, 250, 250]
        
        for c, col_key in enumerate(data.columns):
            cell = table.cell(r, c)
            # Map column name to json key (simple lowercase mapping)
            json_key = col_key.lower()
            val = row_data.get(json_key, "")
            
            cell.fill.solid()
            cell.fill.fore_color.rgb = get_rgb(row_fill)
            
            cell.text_frame.clear() 
            p = cell.text_frame.paragraphs[0]
            p.font.name = Fonts.SANS
            p.font.size = Pt(11)
            p.text = val

            # STATUS LOGIC (No Emojis, just Bold Color)
            if json_key == "status":
                p.font.bold = True
                if "Completed" in val:
                    p.font.color.rgb = get_rgb(Colors.STATUS_GREEN)
                elif "In Progress" in val:
                    p.font.color.rgb = get_rgb(Colors.STATUS_BLUE)
                elif "Pending" in val:
                    p.font.color.rgb = get_rgb(Colors.STATUS_RED)
            else:
                p.font.color.rgb = get_rgb(Colors.TITLE)


# ==========================================
# 🚀 4. USER CONFIG (JSON)
# ==========================================
input_json = {
  "filename": "Supreme_Phase1.pptx",
  "slides": [
    # COVER PAGE
    {
      "type": "cover",
      "title": "Project Report - Biller Management Portal",
      "subtitle": "Bill Payment Platform Enhancement to support Standardized Onboarding",
      "version_date": "Version 1.1 | December 11, 2025",
      "logo_left": "logo_supreme.png",  # Give filename, script creates placeholder if missing
      "logo_right": "logo_newfields.png"
    },
    # TIMELINE PAGE
    {
      "type": "table",
      "title": "Project Timeline & Current Status",
      "banner_title": "Proposed Project Timeline:",
      "banner_text": "The project was planned over a 10-week development cycle. As of December 11, 2025, the project is in Week 11, we are on track with major core development.",
      "columns": ["Phase", "Timeline", "Status", "Notes"],
      "rows": [
        {"phase": "Requirements & UI Design", "timeline": "29th Sep - 12th Oct", "status": "Completed", "notes": "All core UI/UX designs are completed"},
        {"phase": "Onboarding Development", "timeline": "6th Oct - 16 Nov", "status": "Completed", "notes": "Multi-step admin onboarding wizard fully functional"},
        {"phase": "Desktop/mPOS Implementation", "timeline": "13th Oct - 30th Nov", "status": "Completed", "notes": "Admin and Biller self-service web portals"},
        {"phase": "CMS Platform", "timeline": "27th Oct - 7th Dec", "status": "Completed", "notes": "Admin Portal - Complete biller and user management"},
        {"phase": "Reporting - Billers", "timeline": "24th Nov - 8th Dec", "status": "Completed", "notes": "Customer & Bill History, Master Bill Payment Report"},
        {"phase": "User Acceptance Testing", "timeline": "1st Dec - 8th Dec", "status": "In Progress", "notes": "Awaiting stakeholder review"}
      ]
    }
  ]
}

# ==========================================
# 🏁 5. EXECUTE
# ==========================================

try:
    print("⚙️ Generating Phase 1 Deck...")
    config = PresentationConfig(**input_json)
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    for slide_data in config.slides:
        if slide_data.type == "cover":
            render_cover(prs, slide_data)
        elif slide_data.type == "table":
            render_table_slide(prs, slide_data)

    prs.save(config.filename)
    
    # Download Link
    with open(config.filename, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    link = f'<a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" download="{config.filename}" style="background: #0078D4; color: white; padding: 10px 20px; border-radius: 5px; text-decoration: none; font-weight: bold;">⬇️ DOWNLOAD PHASE 1 PPT</a>'
    display(HTML(link))
    print("✅ Done.")

except Exception as e:
    print(f"❌ Error: {e}")